# Lesson 10 Lab — INT8 SmoothQuant and Activation Outliers

**Puzzle:** Can we make activations easier to quantize without changing the floating-point linear layer?

The saved outputs were generated by executing every code cell on the recorded RTX 5090. Run all cells to regenerate the evidence on your own CUDA GPU.

## 0. Predict before running

Write down: (1) the expected direction, (2) the mechanism, (3) the observation that would reverse your prediction, and (4) the evidence level required for the claim.

## 1. Theory — objects and data flow

SmoothQuant operates on matching input channels of activation `X` and weight `W` for a linear layer `Y=XWᵀ`.

### Core mechanism

For positive channel scales `s`, `(X / s)(W · s)ᵀ = XWᵀ`. Choosing `s_j` from activation and weight maxima moves channel difficulty without changing the floating-point function. The exponent `alpha` decides how much range moves toward weights.

In [1]:
from pathlib import Path
import json
import sys
import torch

chapter_rel = Path("chapters/01-mixed-precision-int4")
repo_root = next(
    p for p in [Path.cwd(), *Path.cwd().parents]
    if (p / chapter_rel / "support" / "lab_common.py").exists()
)
sys.path.insert(0, str(repo_root / chapter_rel / "support"))
from lab_common import (base_result, cuda_benchmark, environment_record,
                        error_metrics, require_cuda, save_result,
                        symmetric_quantize)

lesson_dir = repo_root / chapter_rel / "10-smoothquant"
device = require_cuda()
torch.manual_seed(2026 + 10)
environment = environment_record()
print(json.dumps(environment, indent=2))


{
  "gpu": "NVIDIA GeForce RTX 5090",
  "compute_capability": "12.0",
  "gpu_memory_gib": 31.358,
  "python": "3.12.13",
  "torch": "2.12.0",
  "cuda_runtime": "13.0"
}


## 2. Connect theory to the experiment

### Engineering trade-off

Activation ranges become easier for INT8 while weight ranges become harder. The correct objective is combined W8A8 output error and backend performance, not activation amax alone.

### What this code tests

The notebook checks the algebraic invariant before quantizing both sides and comparing output error across alpha values.

**Experiment:** Apply SmoothQuant-style channel scaling to an outlier-heavy linear layer, verify floating-point equivalence, and compare W8A8 reconstruction error over alpha values.

**Declared evidence label:** `numerical-model`. Check that the shapes, controlled variables, and units match the theoretical question before executing.

In [2]:
batch,in_f,out_f=512,512,384; x=torch.randn(batch,in_f,device=device); w=torch.randn(out_f,in_f,device=device)
x[:,::64]*=18; reference=x@w.t(); rows=[]
for alpha in (0.0,0.25,0.5,0.75,1.0):
    ax=x.abs().amax(0).clamp_min(1e-6); aw=w.abs().amax(0).clamp_min(1e-6)
    s=(ax.pow(alpha)/aw.pow(1-alpha)).clamp_min(1e-6); xs=x/s; ws=w*s
    equivalence=(reference-xs@ws.t()).abs().max().item()
    sx=xs.abs().max()/127; sw=ws.abs().max()/127
    qx=torch.round(xs/sx).clamp(-128,127)*sx; qw=torch.round(ws/sw).clamp(-128,127)*sw
    rows.append({"alpha":alpha,"float_equivalence_max_abs":round(equivalence,6),"output_error":error_metrics(reference,qx@qw.t())})
result=base_result(10,"numerical-model"); result.update({"shape":[batch,in_f,out_f],"alpha_sweep":rows,
    "conclusion":"Reciprocal scaling preserved the floating-point layer while changing combined W8A8 error."})


## 3. Inspect the evidence

First verify algebraic equivalence; then compare quantized output error. A lower activation range alone is incomplete evidence.

### Acceptance and rollback gate

Verify floating-point equivalence first, freeze calibration statistics, sweep alpha on calibration data, and accept using held-out output/quality plus native W8A8 evidence.

In [3]:
artifact_path = save_result(result, lesson_dir)
print(json.dumps(result, indent=2, sort_keys=True))
print("Saved: artifacts/rtx5090-result.json")


{
  "alpha_sweep": [
    {
      "alpha": 0.0,
      "float_equivalence_max_abs": 6.1e-05,
      "output_error": {
        "cosine": 0.99824685,
        "mae": 2.62906098,
        "max_abs": 15.94864273,
        "rmse": 3.29818392
      }
    },
    {
      "alpha": 0.25,
      "float_equivalence_max_abs": 4.6e-05,
      "output_error": {
        "cosine": 0.99955261,
        "mae": 1.32819116,
        "max_abs": 7.69526672,
        "rmse": 1.6633786
      }
    },
    {
      "alpha": 0.5,
      "float_equivalence_max_abs": 6.1e-05,
      "output_error": {
        "cosine": 0.99978542,
        "mae": 0.91851175,
        "max_abs": 5.98374176,
        "rmse": 1.15184021
      }
    },
    {
      "alpha": 0.75,
      "float_equivalence_max_abs": 6.1e-05,
      "output_error": {
        "cosine": 0.99956822,
        "mae": 1.30419183,
        "max_abs": 7.36639786,
        "rmse": 1.63480675
      }
    },
    {
      "alpha": 1.0,
      "float_equivalence_max_abs": 4.6e-05,
      "outp

## 4. Explain the result

Outlier migration is useful only when the combined activation-plus-weight quantized path improves under a frozen calibration protocol.

Relate the measured fields back to the mechanism above. Treat the checked-in result as one hardware/software observation, not a universal ranking. The complete derivation, evidence boundary, and primary references are in [`README.md`](README.md).